# Anima — full 90k subject-bucket LoRA (persistent RTX PRO 6000 Blackwell box)

The scaled-up sibling of the preliminary 1k Colab notebook — **same structure, same validated
methodology**, repurposed for the **large `qwen_90k` run** on a **persistent** Linux Blackwell box
(sm_120, 96 GB). What changes vs the 1k notebook:

- **`LIMIT`-capped extraction** (`None` = full ~83k ≈ ~100 GB / 56 shards; or a subset that fits the
  disk — this run uses **40k**) onto a **roomy disk** — set `DATA_ROOT`.
- **Persistent durability:** checkpoints live on local disk; recovery is `--resume_from_checkpoint`,
  not a re-download. HF backup is **optional**, not the lifeline.
- **Long-run config:** `save_every_n_steps`, gradient accumulation, **activation_checkpointing ON**
  (frees VRAM for a bigger `micro_batch`), `compile=true`, `checkpoint_every_n_minutes`.
- **Run detached** (tmux/nohup) so a kernel restart can't kill an ~18 h/epoch job.

Methodology (unchanged — it worked on 1k): `task_1` JSON captions trained **verbatim** in semantic
subject buckets; **before_after** = full VLM phase then full animetimm phase (resumed); dampened
`num_repeats` holds the balance; `llm_adapter_lr = 0` (adapter frozen).

> **License — NON-COMMERCIAL.** Anima and any LoRA from it inherit CircleStone NC + the NVIDIA Open
> Model License (Cosmos derivative). Keep derivatives non-commercial.

**How to run:** top to bottom. §2 has a **one-time install + kernel restart** (skip it if your venv is
already set up). **All real state (`DATA_ROOT`, `REPO`, `HF_HOME`, `sys.path`) is (re-)established in
§3, after the restart** — so a restart never orphans it.

| step | section |
|---|---|
| confirm GPU | §1 |
| clone + install (→ restart) | §2 |
| **(after restart) env + big-disk cache + verify** | §3 |
| HF auth (optional) | §4 |
| download model files | §5 |
| **extract a `LIMIT` subset (40k)** | §6 |
| **build the two phase configs** | §7 |
| cache latents/text-embeds | §8 |
| **train (detached, recommended)** | §9 |
| monitor / resume / get the LoRA | §10 |

## 1 · Confirm the GPU

Expect **RTX PRO 6000 Blackwell** (sm_120) and ~96 GB. bf16 throughout; **no fp8, no flash-attn**
(PyTorch SDPA is correct on sm_120).

In [ ]:
!nvidia-smi

## 2 · Clone the repos and install (→ restart)

Clones the trainer + **diffusion-pipe with submodules** (if missing), then installs torch cu128,
diffusion-pipe's requirements, the bridge deps, and the `[similarity]` extra. **Skip the install cell
+ restart if your venv already has these** — just run the clone cell, then jump to §3.

> The repo location is read from `$ANIMA_REPO` (export it in the shell that launched Jupyter so it
> survives the restart), defaulting to `/workspace/anima-trainer`. Only the small *code* lives here —
> the big *data* goes under `DATA_ROOT`, set in §3.

In [ ]:
import os, subprocess
REPO = os.environ.get("ANIMA_REPO", "/workspace/anima-trainer")   # export ANIMA_REPO to override
DP   = f"{REPO}/external/diffusion-pipe"

def sh(cmd):
    print("$", cmd)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f"command failed: {cmd}")

if not os.path.isfile(f"{REPO}/pyproject.toml"):
    sh(f"git clone https://github.com/AbstractEyes/anima-trainer.git {REPO}")
if not os.path.isfile(f"{DP}/train.py"):
    sh(f"git clone --recurse-submodules https://github.com/tdrussell/diffusion-pipe.git {DP}")
assert os.path.isfile(f"{DP}/train.py"), "diffusion-pipe/train.py missing — clone failed"
print("REPO:", REPO, "| diffusion-pipe:", DP)

In [ ]:
# OPTIONAL — skip if your venv already has torch cu128 + diffusion-pipe deps.
# cu128 torch + reqs + bridge + [similarity]; re-pin datasets<3 and torch LAST so nothing overrides them.
!pip -q install --index-url https://download.pytorch.org/whl/cu128 torch torchvision
!pip -q install -r {DP}/requirements.txt
!pip -q install "huggingface_hub>=0.23" "Pillow>=10" "pyarrow>=14" "sentence-transformers>=2.7,<4" "scikit-learn"
!pip -q install "datasets>=2.19,<3"
!pip -q install --index-url https://download.pytorch.org/whl/cu128 torch torchvision
print("\n✅ installs done. RESTART THE KERNEL so the new torch loads (next cell), then continue at §3.")

**Restart the kernel** so the freshly-installed torch is the one that loads (run the cell below,
or Kernel ▸ Restart). After it restarts, **continue at §3** — do not re-run §2. *(Skip this if you
didn't run the install cell.)*

In [ ]:
# Hard kernel restart so the freshly-installed cu128 torch is the one that loads. Reconnect, then run §3.
import os
os.kill(os.getpid(), 9)

## 3 · (after restart) Environment + big-disk cache + verify

The kernel restart cleared everything, so this cell **(re-)establishes all state** the rest of the
notebook uses — `REPO`/`DP`, the **`DATA_ROOT` big disk** (extraction + latent cache + runs), `HF_HOME`
under it (set **before** any HF import), and `sys.path` (run the package from source). Then it verifies
torch sees Blackwell + cu128 + bf16.

> Set `DATA_ROOT` (and `REPO`) here — or export `ANIMA_DATA_ROOT` / `ANIMA_REPO` in your shell so they
> survive restarts with no editing. Disk need **scales with `LIMIT`** (~150 GB+ for the full ~83k;
> proportionally less for a subset — and you can reclaim the parquet shard cache after §6, see the note there).

In [ ]:
import os, sys
DATA_ROOT = os.environ.get("ANIMA_DATA_ROOT", "/workspace/anima_data")   # >>> EDIT: roomy disk for ALL data
REPO      = os.environ.get("ANIMA_REPO", "/workspace/anima-trainer")     # must match §2
DP        = f"{REPO}/external/diffusion-pipe"

os.makedirs(DATA_ROOT, exist_ok=True)
os.environ["HF_HOME"] = f"{DATA_ROOT}/hf_cache"        # HF shard cache on the big disk (before any HF import)
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
sys.path.insert(0, REPO)                               # import geolip_anima_trainer from source
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

import torch
ok = torch.cuda.is_available()
name = torch.cuda.get_device_name(0) if ok else "NONE"
cap = torch.cuda.get_device_capability(0) if ok else (0, 0)
sm = cap[0] * 10 + cap[1]
print(f"torch {torch.__version__} | cuda {torch.version.cuda} | {name} | sm_{sm} "
      f"| bf16 {torch.cuda.is_bf16_supported() if ok else False}")
print("DATA_ROOT:", DATA_ROOT, "| HF_HOME:", os.environ["HF_HOME"], "| REPO:", REPO)
assert ok, "No CUDA device — start Jupyter from the GPU box."
cb = tuple(int(x) for x in (torch.version.cuda or '0.0').split('.')[:2])
assert cb >= (12, 8), f"torch cuda {torch.version.cuda} < 12.8 — run the §2 install + restart."
if sm < 120:
    print(f"⚠️ sm_{sm}, not Blackwell sm_120 — this notebook is tuned for the RTX PRO 6000 Blackwell.")

In [ ]:
import geolip_anima_trainer as anima
print("geolip_anima_trainer", anima.__version__, "->", anima.__file__)
print(anima.doctor(repo_root=REPO).render())

## 4 · HuggingFace auth (optional)

Only needed for the **optional** checkpoint backup in §10 (and to avoid dataset rate limits). Set
`HF_TOKEN` in the box's environment, or `huggingface-cli login` in a shell. Dataset + model are public.

In [ ]:
import os
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_USER = None
if HF_TOKEN:
    from huggingface_hub import login, whoami
    login(token=HF_TOKEN, add_to_git_credential=True)
    HF_USER = whoami(token=HF_TOKEN)["name"]
    print("logged in as", HF_USER)
else:
    print("no HF_TOKEN -> backups disabled (fine — checkpoints persist on disk). Dataset/model read anonymously.")
BACKUP_REPO = f"{HF_USER}/anima-90k-r64" if HF_USER else None       # MODEL repo: LoRA checkpoints
CACHE_REPO  = f"{HF_USER}/anima-90k-cache" if HF_USER else None     # DATASET repo: frozen data + cache
print("checkpoint backups ->", BACKUP_REPO, "| cache repo ->", CACHE_REPO)

## 5 · Download the Anima model files (~5.6 GB)

Into `models/anima/` under the repo. Returns the `[model]` paths.

In [ ]:
MODELS_DIR = f"{REPO}/models/anima"
paths = anima.download_models(MODELS_DIR, base="base-v1.0")
print("transformer:", paths.transformer_path)
print("vae        :", paths.vae_path)
print("llm        :", paths.llm_path)

## 6 · Get the dataset: resume from HF, or extract + store the index

To **accumulate the cache across sessions** without wasting HF space on images (they already live in
the source parquet, columnar-accessible by id):
- **First session:** extract a `LIMIT`-image subset into the two `before_after` trees. Extraction also
  writes an **`index.jsonl`** — per image: its source id+shard, the pinned bucket placement, and the
  verbatim captions. `cache_push` then stores **only the index** (+ later the cache shards) to a
  private HF *dataset* repo — **never the images or `.txt`**.
- **Resume session:** `cache_pull` downloads the cache + index, then **rebuilds the images
  byte-identically** by columnar-refetching them from the source parquet (`reconstruct_from_index`).
  The rebuilt tree matches the original exactly, so the accumulated cache resumes.

> **⚠️ Pin `DATA_ROOT`.** The cache fingerprint embeds **absolute image paths**, so the rebuild must
> land at the **same absolute `DATA_ROOT`** every session (`reconstruct` hard-errors on a mismatch —
> loud, not a silent wipe). Keep `ANIMA_DATA_ROOT` constant. A Drive-mounted path also lets the rebuilt
> images persist across resets (skips the refetch).

`LIMIT` (`None` = full ~83k) caps accepted images. `[subjects] …` / `[reconstruct] …` lines show progress.

In [ ]:
import os
from pathlib import Path
SUBJECTS_ROOT = f"{DATA_ROOT}/anima_subjects"
LIMIT = 40000   # disk-limited subset for THIS run (full ~83k won't fit). None = everything.

# SUBJ_CFG is needed by §7 (build_mode_tomls reads caption_mode) in BOTH branches.
SUBJ_CFG = anima.SubjectBucketConfig(
    repo="AbstractPhil/diffusion-pretrain-set-ft1", config="qwen_90k",
    out_root=SUBJECTS_ROOT, limit=LIMIT,
    require_audit_approved=True, require_age_pass=False,   # age_classifier_pass unpopulated -> gate off
    caption_mode=anima.CaptionMode.BEFORE_AFTER,
    use_semantic=True, similarity_model="sentence-transformers/all-MiniLM-L6-v2",
    min_bucket_size=10, min_final_group_size=8, keep_small=True, split_oversized=True,
)

def _has_cache(root):
    return Path(root).is_dir() and any(Path(root).rglob("metadata.db"))

RESUME = _has_cache(SUBJECTS_ROOT)
if not RESUME and CACHE_REPO:                          # restore a prior session: cache+index, then rebuild images
    try:
        anima.cache_pull(SUBJECTS_ROOT, CACHE_REPO, token=HF_TOKEN)   # downloads cache+index, refetches images
        RESUME = _has_cache(SUBJECTS_ROOT)
        print("pulled cache+index + rebuilt images ->", SUBJECTS_ROOT, "| resume:", RESUME)
    except Exception as e:
        print("no prior cache to pull (", repr(e), ") -> will extract fresh")

if not RESUME:                                         # FIRST session: extract, then store the INDEX on HF
    rep = anima.export_subject_buckets(SUBJ_CFG)
    print(f"mode={rep['caption_mode']} accepted={rep['accepted_images']} dropped={rep['dropped_images']} "
          f"raw_subjects={rep['raw_subjects']} -> buckets={rep['n_final_buckets']} cap={rep['max_bucket_size']}")
    print("index:", rep["index_path"])
    for k, v in list(rep["final_buckets"].items())[:20]:
        print(f"  {k:<34}{v}")
    if CACHE_REPO:
        anima.cache_push(SUBJECTS_ROOT, CACHE_REPO, token=HF_TOKEN,
                         commit_message="store index (pre-cache)")    # index only — no images, no cache yet
        print("stored index ->", f"https://huggingface.co/datasets/{CACHE_REPO}")
else:
    print("RESUME: reusing the restored dataset at", SUBJECTS_ROOT,
          "(images rebuilt from source -> byte-identical -> stable cache fingerprint)")

## 7 · Build the two phase configs (90k long-run knobs)

Two balanced dataset tomls + one lora toml per phase, with the **long-run** RunConfig:
- **`save_every_n_steps`** — step-based saving (epoch saving is ~18 h apart). One *step* =
  `micro_batch × grad_accum` examples.
- **`gradient_accumulation_steps`** — grow the effective batch with **no extra VRAM**.
- **`activation_checkpointing=True`** — frees ~50–60 GB so `micro_batch` can go up (≈25–33% slower per
  step, bigger batch + headroom; the right 96 GB shape for a long run).
- **`compile=True`** (~+10–25%, amortized over days) and **`checkpoint_every_n_minutes`** (resume-state).

In [ ]:
import os
from pathlib import Path

# ---- long-run tunables (full 90k) ------------------------------------------------
RANK                 = 64
LR                   = 2e-5
RES                  = 1024
EPOCHS_VLM           = 3              # 90k is huge per epoch — a few go a long way; tune to your budget
EPOCHS_ANIMETIMM     = 2
MICRO_BATCH          = 16             # fits with activation_checkpointing ON (tune toward ~90 GB)
GRAD_ACCUM           = 4              # effective batch = MICRO_BATCH * GRAD_ACCUM = 64
SAVE_EVERY_N_STEPS   = 500            # 1 step = MICRO_BATCH*GRAD_ACCUM examples (=64 -> ~32k imgs/save)
SAVE_EVERY_N_EPOCHS  = 1             # backstop (diffusion-pipe needs >=1 save_* key)
CKPT_EVERY_N_MIN     = 30            # full resume-state cadence (for --resume_from_checkpoint)
WARMUP_STEPS         = 200
ACT_CKPT             = True           # ON for the long run (frees VRAM for the bigger batch)
COMPILE              = True
BALANCE_ALPHA        = 0.5
MAX_REPEATS          = 8
CAP_MULT             = 1.25
CACHING_BATCH        = 8
MAP_NUM_PROC         = os.cpu_count() or 16
OUTPUT_DIR           = f"{DATA_ROOT}/runs/anima_90k"
CONFIGS_DIR          = f"{REPO}/configs/full90k"
# ----------------------------------------------------------------------------------

ds_tomls = anima.build_mode_tomls(SUBJECTS_ROOT, SUBJ_CFG, configs_dir=CONFIGS_DIR,
                                  resolutions=[RES], balance_alpha=BALANCE_ALPHA,
                                  cap_mult=CAP_MULT, max_repeats=MAX_REPEATS)
ds_tomls = {Path(t).stem: str(t) for t in ds_tomls}
assert "dataset_vlm" in ds_tomls and "dataset_animetimm" in ds_tomls, ds_tomls
print("dataset tomls:", ds_tomls)

model = anima.ModelConfig(transformer_path=paths.transformer_path, vae_path=paths.vae_path,
                          llm_path=paths.llm_path, llm_adapter_lr=0.0)   # adapter FROZEN

def _phase_lora(phase, ds_toml, epochs):
    run = anima.RunConfig(
        output_dir=f"{OUTPUT_DIR}/{phase}", epochs=epochs,
        micro_batch_size_per_gpu=MICRO_BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        save_every_n_steps=SAVE_EVERY_N_STEPS, save_every_n_epochs=SAVE_EVERY_N_EPOCHS,
        checkpoint_every_n_minutes=CKPT_EVERY_N_MIN, warmup_steps=WARMUP_STEPS,
        activation_checkpointing=ACT_CKPT, compile=COMPILE,
        caching_batch_size=CACHING_BATCH, map_num_proc=MAP_NUM_PROC)
    cfg = anima.TrainConfig(run=run, model=model, adapter=anima.AdapterConfig(rank=RANK),
                            optimizer=anima.OptimizerConfig(lr=LR),
                            dataset=anima.DatasetConfig(resolutions=[RES]),
                            dataset_toml_path=ds_toml)
    anima.validate(cfg)
    p = Path(CONFIGS_DIR) / f"lora_{phase}.toml"
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(anima.render_lora_toml(cfg), encoding="utf-8")
    return str(p)

LORA_VLM       = _phase_lora("vlm",       ds_tomls["dataset_vlm"],       EPOCHS_VLM)
LORA_ANIMETIMM = _phase_lora("animetimm", ds_tomls["dataset_animetimm"], EPOCHS_ANIMETIMM)
print("phase 1 (vlm)      :", LORA_VLM)
print("phase 2 (animetimm):", LORA_ANIMETIMM)

## 8 · Cache latents + text embeddings (both phases) — with HF backup + resume

`--cache_only` precomputes VAE latents + Qwen-3 text embeds onto `DATA_ROOT`. The latent counter ticks
up live as shards finalize (decode-bound; `map_num_proc` is the lever). Two safety nets now:
- **Periodic push to `CACHE_REPO`** every shard-finalize / 30 min, **plus a guaranteed final push even
  if the run crashes** — so a Colab reset never loses more than the in-progress ≤10 GB shard (no more
  8-hour losses). Pushes are **cache-only** (the dataset was frozen in §6); `backup_root=SUBJECTS_ROOT`
  keeps the repo layout aligned with the freeze.
- **`trust_cache=RESUME`** — on a resume session, the pulled cache's metadata loads without
  re-validation and already-cached items are skipped (`[CACHE] Existing cache …`), so it picks up where
  it left off. The first cell previews the command.

In [ ]:
anima.cache(LORA_VLM, repo_root=REPO, num_gpus=1, dry_run=True)   # preview the cache command

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
for phase, lora in (("vlm", LORA_VLM), ("animetimm", LORA_ANIMETIMM)):
    print(f"\n=== caching {phase} ===")
    rc = anima.cache(lora, repo_root=REPO, num_gpus=1, progress=True, progress_interval=60,
                     log_path=f"{OUTPUT_DIR}/cache_{phase}.log",
                     backup_repo=CACHE_REPO, backup_root=SUBJECTS_ROOT, backup_interval=1800,
                     backup_token=HF_TOKEN, trust_cache=RESUME)   # periodic + final HF push; resume-aware
    print(f"cache[{phase}] exit code:", rc)
if CACHE_REPO:
    anima.cache_push(SUBJECTS_ROOT, CACHE_REPO, token=HF_TOKEN, commit_message="post-cache full")
    print("pushed full cache ->", f"https://huggingface.co/datasets/{CACHE_REPO}")

## 9 · Train both phases — run DETACHED (recommended)

An ~18 h/epoch job must survive a kernel restart, so **run it detached** in tmux/screen via the CLI —
this preserves the phase-1 → phase-2 adapter handoff that `train-before-after` resolves at runtime.
Print the command, then paste it into a terminal on the box:

In [ ]:
DETACHED = (f"cd {REPO} && ANIMA_DATA_ROOT={DATA_ROOT} nohup python -m geolip_anima_trainer.cli "
            f"train-before-after --lora-vlm {LORA_VLM} --lora-animetimm {LORA_ANIMETIMM} "
            f"--repo-root {REPO} --num-gpus 1 > {OUTPUT_DIR}/train.log 2>&1 &")
print("# run this in a tmux/screen shell on the box:\n")
print(DETACHED)
print(f"\n# then watch:   tail -f {OUTPUT_DIR}/train.log")
print(f"# tensorboard:  tensorboard --logdir {OUTPUT_DIR}")

**Or run in-notebook** (background thread) — fine for a shorter session, but a kernel restart
kills it, so prefer the detached command above for the real multi-day run. Each phase logs to
`OUTPUT_DIR/{vlm,animetimm}.log`.

In [ ]:
import threading
_train = {}
def _run():
    try:
        _train["rc"] = anima.train_before_after(LORA_VLM, LORA_ANIMETIMM, repo_root=REPO,
                                                num_gpus=1, configs_dir=CONFIGS_DIR, log_dir=OUTPUT_DIR)
    except Exception as e:
        _train["error"] = repr(e); raise

train_thread = threading.Thread(target=_run, name="anima-train", daemon=True)
train_thread.start()
print("training thread started | logs:", f"{OUTPUT_DIR}/vlm.log -> {OUTPUT_DIR}/animetimm.log")

## 10 · Monitor, resume, get the LoRA

**Watch progress** (works for the detached run or the in-notebook thread):

In [ ]:
import subprocess
from pathlib import Path

def tail(n=40):
    for cand in (f"{OUTPUT_DIR}/animetimm.log", f"{OUTPUT_DIR}/vlm.log", f"{OUTPUT_DIR}/train.log"):
        if Path(cand).exists():
            print(f"--- {Path(cand).name} ---")
            print(subprocess.run(["tail", "-n", str(n), cand], capture_output=True, text=True).stdout)
            return
    print("no log yet")

tail()

**Resume after an interruption** (checkpoints are on disk) — drive each phase with `resume`
(passes `--resume_from_checkpoint`, loading the latest `checkpoint_every_n_minutes` state):
```python
anima.train(LORA_VLM,       repo_root=REPO, num_gpus=1, resume=True)   # finish the VLM phase
anima.train(LORA_ANIMETIMM, repo_root=REPO, num_gpus=1, resume=True)   # ...then the animetimm phase
```

**Get the LoRA:** the final adapter is the newest `OUTPUT_DIR/animetimm/<timestamp>/epochN/` (or the
step-save dir) — ComfyUI-format safetensors + PEFT config; drop the safetensors into
`ComfyUI/models/loras/`. The `vlm/...` checkpoints are the intermediate phase-1 adapter.

**Optional HF backup** (persistent disk is the primary durability; this is a slow heartbeat — run it
whenever, or on a cron). Needs `HF_TOKEN` from §4:

In [ ]:
def backup_latest(repo_id=BACKUP_REPO, output_dir=None, token=HF_TOKEN):
    output_dir = output_dir or OUTPUT_DIR
    if not (repo_id and token):
        print("no HF_TOKEN/BACKUP_REPO -> skipping (checkpoints are safe on disk)"); return None
    from huggingface_hub import HfApi, create_repo
    runs = [d.parent for d in Path(output_dir).rglob("epoch*") if d.is_dir() and any(d.glob("*.safetensors"))]
    if not runs:
        print("no checkpoints yet under", output_dir); return None
    run = max(runs, key=lambda d: d.stat().st_mtime)
    rel = run.relative_to(Path(output_dir)).as_posix()           # e.g. animetimm/<timestamp>
    create_repo(repo_id, token=token, repo_type="model", private=True, exist_ok=True)
    HfApi(token=token).upload_folder(folder_path=str(run), repo_id=repo_id, repo_type="model",
                                     path_in_repo=f"runs/{rel}",
                                     ignore_patterns=["global_step*/*", "global_step*/**"],
                                     commit_message=f"backup :: {rel}")
    print("backed up ->", f"https://huggingface.co/{repo_id}/tree/main/runs/{rel}")

# backup_latest()   # uncomment to push the newest checkpoint now

**Notes**
- `before_after` (validated on 1k) is the recommended recipe; the **animetimm adapter is the final LoRA**.
- Keep `BALANCE_ALPHA=0.5` / `MAX_REPEATS=8` (the dampening that held the balance) and `llm_adapter_lr=0`
  unless deliberately A/B-testing the adapter (ceiling 5e-6 — `validate()` hard-errors above it).
- `save_every_n_steps` counts **optimizer steps**; lower it for finer checkpoints. `activation_checkpointing`
  ON is what lets `MICRO_BATCH` be large here — the 1k run left it off for raw speed.

> **License reminder:** the LoRA is a non-commercial Anima derivative (CircleStone NC + NVIDIA Open
> Model License).